# 🔍 SEC EDGAR — Exploratory Data Analysis
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-29

---

### Notebook Objectives
1. Analyze filing trends over time
2. Explore industry distribution
3. Discover financial tags relevant to ratio analysis
4. Compare Pre vs Post COVID filing behavior
5. Validate sample company financials
6. Define micro business questions

### Tables Used
- `edgar_sub_all` → 192,059 rows — filing metadata
- `edgar_num_all` → 88,170,247 rows — financial numbers

## Column Descriptions

### `edgar_sub_all` — Filing Metadata

| Column | Description |
|--------|-------------|
| `adsh` | Accession Number — unique identifier for each SEC filing |
| `cik` | Central Index Key — unique identifier for each company |
| `name` | Company name as registered with SEC |
| `sic` | Standard Industrial Classification code — identifies industry sector |
| `countryba` | Country of business address |
| `stprba` | State or province of business address |
| `cityba` | City of business address |
| `zipba` | Zip code of business address |
| `bas1` | Business address street line 1 |
| `filed` | Date the filing was submitted to SEC |
| `period` | COVID period label — Pre-COVID, COVID-Impact, Post-COVID-Recovery |
| `source_year` | Year the filing was ingested from |
| `source_quarter` | Quarter the filing was ingested from |
| `ingestion_timestamp` | Timestamp when record was ingested into Delta table |
| `pipeline_run_id` | Unique ID of the ingestion pipeline run |

### `edgar_num_all` — Financial Numbers

| Column | Description |
|--------|-------------|
| `adsh` | Accession Number — joins to edgar_sub_all |
| `tag` | XBRL financial tag name — identifies the financial metric |
| `version` | XBRL taxonomy version the tag belongs to |
| `ddate` | Date of the financial data point (end of reporting period) |
| `qtrs` | Number of quarters represented — 0=instant, 1=quarterly, 4=annual |
| `uom` | Unit of measure — USD, shares, pure ratio etc |
| `value` | Numeric value of the financial data point |
| `segments` | Business segment breakdown — NULL means consolidated company level |
| `source_year` | Year the data was ingested from |
| `source_quarter` | Quarter the data was ingested from |
| `ingestion_timestamp` | Timestamp when record was ingested into Delta table |
| `pipeline_run_id` | Unique ID of the ingestion pipeline run |

### Key Relationships
| Relationship | Details |
|-------------|---------|
| Join key | `edgar_sub_all.adsh = edgar_num_all.adsh` |
| Consolidated filter | `edgar_num_all.segments IS NULL` |
| Period classification | Pre-COVID (2018-2019), COVID-Impact (Q1-Q2 2020), Post-COVID-Recovery (Q3 2020-2024) |
| Annual figures | `qtrs = 4` |
| Quarterly figures | `qtrs = 1` |
| Point-in-time figures | `qtrs = 0` |

## Step 1 — Environment Setup

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark session ready | Version: {spark.version}")

## Step 2 — Filing Trends Over Time
How did filing volumes change across 2018 → 2024?
Are there anomalies during COVID quarters?

In [0]:
filing_trends = spark.sql("""
    SELECT 
        source_year,
        source_quarter,
        period,
        COUNT(DISTINCT cik) AS unique_companies,
        COUNT(DISTINCT adsh) AS total_filings
    FROM edgar_sub_all
    GROUP BY source_year, source_quarter, period
    ORDER BY source_year, source_quarter
""")

display(filing_trends)

## Step 3 — Industry Distribution
3.a Which industries dominate SEC filings?
Top 20 SIC codes by unique company count across all periods.

In [0]:
industry_dist = spark.sql("""
    SELECT 
        sic,
        COUNT(DISTINCT cik) AS unique_companies,
        COUNT(DISTINCT adsh) AS total_filings
    FROM edgar_sub_all
    WHERE sic IS NOT NULL
    GROUP BY sic
    ORDER BY unique_companies DESC
    LIMIT 20
""")

display(industry_dist)

3.b Understanding the breadth and distribution of industries in the dataset

In [0]:
unique_sic = spark.sql("""
    SELECT 
        COUNT(DISTINCT sic) AS total_unique_sic_codes,
        COUNT(DISTINCT cik) AS total_unique_companies,
        ROUND(COUNT(DISTINCT cik) / COUNT(DISTINCT sic), 0) AS avg_companies_per_sic
    FROM edgar_sub_all
    WHERE sic IS NOT NULL
""")

display(unique_sic)

3.c Distribution of companies per SIC — are filings concentrated in few industries or spread out

In [0]:
sic_distribution = spark.sql("""
    SELECT 
        company_count_bucket,
        COUNT(*) AS num_sic_codes
    FROM (
        SELECT 
            sic,
            COUNT(DISTINCT cik) AS company_count,
            CASE 
                WHEN COUNT(DISTINCT cik) = 1 THEN '1 company'
                WHEN COUNT(DISTINCT cik) BETWEEN 2 AND 5 THEN '2-5 companies'
                WHEN COUNT(DISTINCT cik) BETWEEN 6 AND 20 THEN '6-20 companies'
                WHEN COUNT(DISTINCT cik) BETWEEN 21 AND 100 THEN '21-100 companies'
                ELSE '100+ companies'
            END AS company_count_bucket
        FROM edgar_sub_all
        WHERE sic IS NOT NULL
        GROUP BY sic
    )
    GROUP BY company_count_bucket
    ORDER BY num_sic_codes DESC
""")

display(sic_distribution)

## Step 4 — Financial Tags Discovery
Which financial metrics are most commonly reported?
Filters to consolidated figures only (segments IS NULL).
Identifies ratio-relevant tags for downstream SQL analysis.

In [0]:
financial_tags = spark.sql("""
    SELECT 
        tag,
        COUNT(*) AS occurrences,
        COUNT(DISTINCT adsh) AS unique_filings,
        ROUND(AVG(TRY_CAST(value AS DOUBLE)), 2) AS avg_value
    FROM edgar_num_all
    WHERE tag IN (
        'Assets',
        'Liabilities',
        'StockholdersEquity',
        'NetIncomeLoss',
        'Revenues',
        'EarningsPerShareBasic',
        'EarningsPerShareDiluted',
        'CashAndCashEquivalentsAtCarryingValue',
        'LongTermDebt',
        'OperatingIncomeLoss',
        'GrossProfit',
        'CurrentAssets',
        'CurrentLiabilities',
        'RetainedEarningsAccumulatedDeficit',
        'CommonStockSharesOutstanding'
    )
    GROUP BY tag
    ORDER BY occurrences DESC
""")

display(financial_tags)

## Step 5 — Pre vs Post COVID Comparison
Did COVID fundamentally change filing behavior?
Compares unique companies, total filings, and industries across periods.

In [0]:
covid_comparison = spark.sql("""
    SELECT 
        period,
        COUNT(DISTINCT cik) AS unique_companies,
        COUNT(DISTINCT adsh) AS total_filings,
        COUNT(DISTINCT sic) AS industries_represented,
        COUNT(DISTINCT source_year) AS years_covered
    FROM edgar_sub_all
    GROUP BY period
    ORDER BY period
""")

display(covid_comparison)